# Release Replay Batches (`scripts/03_release_replay.ipynb`)

Release prepared daily batches through **January 28, 2019** (`initial_history`: Jan 1–14 and `rule_development`: Jan 15–28) into `/Volumes/workspace/sentinel_sources/replay_landing`.

- **Skip already released dates**: Never overwrite existing landing files so Auto Loader incremental ingestion state remains valid.
- **Manifest verification**: Verify every daily batch count in staging and landing against `/Volumes/workspace/sentinel_sources/replay_staging/replay_v1/manifest`.
- **Evaluation holdout**: Keep **January 29–February 11, 2019** (`evaluation` period) unreleased in staging.

In [ ]:
from pyspark.sql import functions as F

# Release initial_history (Jan 1-14) and rule_development (Jan 15-28).
# Keep evaluation batches (Jan 29 onward) unreleased in staging.
release_end_date = "2019-01-28"
evaluation_start_date = "2019-01-29"

staging_root = (
    "/Volumes/workspace/sentinel_sources/"
    "replay_staging/replay_v1/transactions"
)
manifest_path = (
    "/Volumes/workspace/sentinel_sources/"
    "replay_staging/replay_v1/manifest"
)
landing_root = (
    "/Volumes/workspace/sentinel_sources/replay_landing"
)

# Load expected daily counts for January 1-28 from the saved manifest.
manifest = spark.read.parquet(manifest_path)
target_manifest = (
    manifest
    .filter(F.col("replay_batch") <= F.lit(release_end_date).cast("date"))
    .orderBy("replay_batch")
)

target_rows = target_manifest.collect()
assert len(target_rows) == 28, (
    f"Expected 28 daily batches for Jan 1-28; found {len(target_rows)}"
)

# Check which daily batch folders are already present in landing.
existing_folders = {
    item.name.rstrip("/")
    for item in dbutils.fs.ls(landing_root)
}

newly_released_dates = []
skipped_existing_dates = []

for row in target_rows:
    batch_date = str(row["replay_batch"])
    expected_count = int(row["expected_transaction_count"])
    folder_name = f"replay_batch={batch_date}"
    source_path = f"{staging_root}/{folder_name}"
    destination_path = f"{landing_root}/{folder_name}"

    if folder_name in existing_folders:
        # Never overwrite an already released landing folder.
        skipped_existing_dates.append(batch_date)
    else:
        # Verify the staged batch against the manifest before copying.
        staged_batch = spark.read.parquet(source_path)
        staged_count = staged_batch.count()
        assert staged_count == expected_count, (
            f"Staged count mismatch for {batch_date}: "
            f"expected {expected_count:,}, found {staged_count:,}"
        )
        assert "is_fraud" not in staged_batch.columns, (
            f"Fraud labels must not enter landing ({batch_date})"
        )

        copied = dbutils.fs.cp(source_path, destination_path, recurse=True)
        assert copied, f"Batch copy failed for {batch_date}"
        newly_released_dates.append(batch_date)

    # Verify the landing folder count against the manifest.
    released_batch = spark.read.parquet(destination_path)
    released_count = released_batch.count()
    assert released_count == expected_count, (
        f"Landing count mismatch for {batch_date}: "
        f"expected {expected_count:,}, found {released_count:,}"
    )

print(f"Skipped already released dates ({len(skipped_existing_dates)}): {skipped_existing_dates}")
print(f"Newly released dates ({len(newly_released_dates)}): {newly_released_dates}")

In [ ]:
# Verify all landing batches against the manifest and confirm evaluation holdout.
landing_df = spark.read.parquet(landing_root)
assert "is_fraud" not in landing_df.columns, "Fraud labels found in landing"

unreleased_eval_in_landing = (
    landing_df
    .filter(F.col("replay_batch") >= F.lit(evaluation_start_date).cast("date"))
    .count()
)
assert unreleased_eval_in_landing == 0, (
    f"Evaluation batches (>= {evaluation_start_date}) must remain unreleased"
)

landing_daily_counts = (
    landing_df
    .groupBy("replay_batch")
    .agg(F.count("*").alias("landing_transaction_count"))
)

comparison = target_manifest.join(
    landing_daily_counts,
    on="replay_batch",
    how="full",
)

mismatches = comparison.filter(
    F.col("expected_transaction_count").isNull()
    | F.col("landing_transaction_count").isNull()
    | (F.col("expected_transaction_count") != F.col("landing_transaction_count"))
)
assert mismatches.count() == 0, "Landing daily counts differ from the manifest"

expected_total = target_manifest.agg(F.sum("expected_transaction_count")).collect()[0][0]
landing_total = landing_df.count()
assert landing_total == expected_total, (
    f"Expected {expected_total:,} total landing transactions; found {landing_total:,}"
)

print(f"Verified {len(target_rows)} released daily batches in landing: {landing_total:,} transactions.")
print("January 29 onward remains unreleased for evaluation.")

display(
    comparison
    .groupBy("period")
    .agg(
        F.count("*").alias("released_batches"),
        F.sum("expected_transaction_count").alias("expected_transactions"),
        F.sum("landing_transaction_count").alias("landing_transactions"),
    )
    .orderBy("period")
)